
# Apache Iceberg 文件布局详解

Apache Iceberg 通过精心设计的**分层元数据结构**，将海量数据文件组织成一张逻辑上统一、物理上高效、支持 ACID 事务的分析型表。其文件布局是实现高性能查询、时间旅行、并发写入等核心能力的基础。

本文将深入解析 Iceberg 的文件层级结构、各组件作用及其协同工作机制。

---

## 🧱 核心文件层级结构

Iceberg 表的数据并非简单地以目录形式堆砌，而是由一套**自上而下、逐层引用**的元数据文件构成。整体结构如下：

```
<表根目录>/
├── metadata/
│   ├── <uuid>.metadata.json        ← 表元数据 (Table Metadata)
│   ├── snap-<id>-<uuid>.avro       ← Manifest List
│   └── <uuid>.avro                 ← Manifest File (多个)
└── data/
    ├── <uuid>.parquet              ← 数据文件 (Data File, 多个)
    └── <uuid>-deletes.parquet      ← 删除文件 (Delete File, V2+)
```

> 💡 **关键点**：所有路径均为随机 UUID，**禁止直接通过文件系统路径访问数据**！必须通过 Spark/Flink/Trino 等引擎经由 Catalog 访问。

---

## 🔍 各层文件详解

### 1. **Table Metadata (`*.metadata.json`)**
- **作用**：表的“总控中心”，存储当前快照（Snapshot）、Schema、分区规范、属性等全局信息。
- **特点**：
  - 每次 DDL 或写入操作成功后，会生成**新版本**的 metadata 文件。
  - 通过 `previous-metadata-file` 字段形成**版本链**，支持回溯。
- **位置**：位于 `metadata/` 目录下，最新版本通常通过 `version-hint.text` 文件指向。

### 2. **Snapshot (快照)**
- **作用**：代表表在某一时刻的**完整状态**，是时间旅行和 ACID 的基础。
- **存储方式**：快照本身不直接存储为文件，其 ID 和元数据（如时间戳、操作类型）记录在 `metadata.json` 中。
- **关联**：每个 Snapshot 指向一个 **Manifest List**。

### 3. **Manifest List (`snap-<id>-<uuid>.avro`)**
- **作用**：快照的“目录索引”，列出该快照包含的所有 **Manifest File**。
- **内容**：
  - Manifest 文件路径
  - 分区范围摘要（用于分区裁剪）
  - 新增/删除文件计数
- **优势**：查询时可先读取 Manifest List，快速判断哪些 Manifest 可能包含目标数据。

### 4. **Manifest File (`<uuid>.avro`)**
- **作用**：数据文件的“详细清单”，是**高效过滤的核心**。
- **每条记录包含**：
  - 数据文件路径（`data/<uuid>.parquet`）
  - 分区值（内部编码）
  - 列统计信息（Min/Max 值、Null Count）
  - 文件大小、行数
  - 删除文件引用（V2+）
- **谓词下推**：查询引擎利用 Min/Max 统计信息，在**不读取任何数据文件**的情况下剪枝掉无关文件。

### 5. **Data File (`<uuid>.parquet / .orc / .avro`)**
- **作用**：实际存储业务数据的列式文件。
- **格式**：支持 Parquet（最常用）、ORC、Avro。
- **特点**：
  - 文件内**不包含分区列的物理值**（得益于隐藏分区）。
  - 支持按列压缩、编码，提升 I/O 效率。

### 6. **Delete File (`*-deletes.parquet`) — V2 特性**
- **作用**：实现**行级删除**，避免重写整个数据文件。
- **类型**：
  - **Position Delete**：通过文件路径 + 行号标记删除。
  - **Equality Delete**：通过主键值匹配删除（类似 CDC）。
- **查询时合并**：引擎在读取 Data File 时，自动应用对应的 Delete File 进行过滤。

---

## 🔄 文件布局如何支撑核心能力？

| 能力 | 依赖的文件布局机制 |
|------|------------------|
| **ACID 事务** | 原子提交新 `metadata.json` + 快照隔离 |
| **时间旅行** | 保留历史 `metadata.json` 和未被 GC 的 Snapshot |
| **高效扫描** | Manifest File 中的列统计信息 + 分区摘要 → 谓词下推 |
| **隐藏分区** | 分区值仅存于 Manifest，数据文件无冗余；查询自动转换 |
| **Schema 演进** | Schema 存于 `metadata.json`，通过 Column ID 关联旧文件 |
| **行级删除 (V2)** | Delete File 与 Data File 解耦，独立存储删除标记 |

---

## 🛠️ 文件布局维护与优化

由于 Iceberg **不自动合并小文件或清理垃圾**，需定期执行维护任务：

```sql
-- 1. 合并小文件
CALL catalog.system.rewrite_data_files(
  table => 'db.table',
  options => map('target-file-size-bytes', '134217728') -- 128MB
);

-- 2. 清理过期快照
CALL catalog.system.expire_snapshots(
  table => 'db.table',
  retain_last => 10,
  older_than => interval '3 days'
);

-- 3. 删除孤儿文件
CALL catalog.system.delete_orphan_files(
  table => 'db.table',
  older_than => interval '3 days'
);
```

> ⚠️ **注意**：`older_than` 时间窗口务必保守（建议 ≥ 3 天），防止误删正在写入的文件。

---

## 📌 最佳实践总结

1. **写入时排序**：对高频过滤列进行 `Z-Order` 或 `Sort`，提升 Manifest 中 Min/Max 的区分度。
2. **合理分区粒度**：流式用 `hours()`，批量用 `days()`，避免过细导致 Manifest 爆炸。
3. **监控文件分布**：定期查询 `table.files` 系统表，检查小文件比例。
4. **自动化维护**：通过 Airflow/DolphinScheduler 定时调度 `rewrite`、`expire`、`delete_orphan`。
5. **绝不直连文件系统**：Iceberg 是**表格式**，不是文件夹！所有操作必须通过计算引擎。

---

通过这套清晰、可扩展、元数据驱动的文件布局，Apache Iceberg 成功将数据湖从“一堆文件”升级为“真正的数据库表”，为现代大数据分析奠定了坚实基础。